In [1]:
import os
import glob
import numpy as np
import pandas as pd

# ============================================================
# User settings
# ============================================================
TYPE_AL = 1
TYPE_O  = 2
TYPE_OI = 3
TYPE_OV_CORE_AL = 4

MIN_DIST = 1.5
N_EVENTS_PER_SAMPLE = 10
FILE_PATTERN = "amorphous_300K_sample*_full*"

OUT_DIR = "OvOi_events"
SUMMARY_CSV = "OvOi_event_summary.csv"

os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================
# Utilities
# ============================================================
def minimum_image(vec, box):
    return vec - box * np.round(vec / box)

def deepcopy_atoms(atoms):
    return [a.copy() for a in atoms]

def is_numeric_line(s):
    s = s.strip()
    if not s:
        return False
    return s.split()[0].lstrip("-").isdigit()

# ============================================================
# Read LAMMPS data (Atoms # full)
# ============================================================
def read_lammps_data(filename):
    with open(filename) as f:
        lines = f.readlines()

    def find_bounds(axis):
        key = f"{axis}lo {axis}hi"
        for l in lines:
            if key in l:
                lo, hi = map(float, l.split()[:2])
                return lo, hi
        raise RuntimeError(f"[{filename}] box bounds not found for {axis}")

    xlo, xhi = find_bounds("x")
    ylo, yhi = find_bounds("y")
    zlo, zhi = find_bounds("z")

    box = np.array([xhi - xlo, yhi - ylo, zhi - zlo])
    origin = np.array([xlo, ylo, zlo])

    atoms_header = None
    for i, l in enumerate(lines):
        if l.strip().startswith("Atoms"):
            atoms_header = i
            break
    if atoms_header is None:
        raise RuntimeError("Atoms section not found")

    atoms_start = None
    for i in range(atoms_header + 1, len(lines)):
        if is_numeric_line(lines[i]):
            atoms_start = i
            break
    if atoms_start is None:
        raise RuntimeError("Atoms data not found")

    atoms = []
    for i in range(atoms_start, len(lines)):
        s = lines[i].strip()
        if not is_numeric_line(s):
            break
        t = s.split()
        atoms.append({
            "id": int(t[0]),
            "mol": int(t[1]),
            "type": int(t[2]),
            "q": float(t[3]),
            "x": float(t[4]),
            "y": float(t[5]),
            "z": float(t[6]),
            "ix": int(t[7]),
            "iy": int(t[8]),
            "iz": int(t[9]),
        })

    return atoms, box, origin, lines

# ============================================================
# REMOVE ENTIRE Velocities section (robust)
# ============================================================
def remove_entire_velocities_section(lines):
    out = []
    i = 0
    n = len(lines)

    while i < n:
        s = lines[i].strip()

        if s.startswith("Velocities"):
            i += 1
            while i < n:
                ss = lines[i].strip()
                if ss == "":
                    i += 1
                    continue
                if ss.split()[0].lstrip("-").isdigit():
                    i += 1
                    continue
                break
            continue

        out.append(lines[i])
        i += 1

    return out

# ============================================================
# Enforce header / Masses / Atoms # full
# ============================================================
def enforce_header_and_masses(lines, natoms):
    for i, l in enumerate(lines):
        if l.strip().endswith("atoms"):
            lines[i] = f"{natoms} atoms\n"
        if l.strip().endswith("atom types"):
            lines[i] = "4 atom types\n"

    masses_idx = None
    for i, l in enumerate(lines):
        if l.strip().startswith("Masses"):
            masses_idx = i
            break
    if masses_idx is None:
        raise RuntimeError("Masses section not found")

    mstart = None
    for i in range(masses_idx + 1, len(lines)):
        if is_numeric_line(lines[i]):
            mstart = i
            break
    if mstart is None:
        raise RuntimeError("Masses numeric lines not found")

    mend = mstart
    for i in range(mstart, len(lines)):
        if not is_numeric_line(lines[i]):
            mend = i
            break

    lines[mstart:mend] = [
        "1 26.9815\n",
        "2 15.9994\n",
        "3 15.9994\n",
        "4 26.9815\n",
    ]

    for i, l in enumerate(lines):
        if l.strip().startswith("Atoms"):
            lines[i] = "Atoms # full\n"
            break

    return lines

# ============================================================
# Write data file
# ============================================================
def write_lammps_data(template_lines, atoms, outname):
    lines = remove_entire_velocities_section(template_lines)
    lines = enforce_header_and_masses(lines, len(atoms))

    atoms_header = None
    for i, l in enumerate(lines):
        if l.strip().startswith("Atoms"):
            atoms_header = i
            break
    if atoms_header is None:
        raise RuntimeError("Atoms header missing")

    atoms_start = None
    for i in range(atoms_header + 1, len(lines)):
        if is_numeric_line(lines[i]):
            atoms_start = i
            break
    if atoms_start is None:
        atoms_start = atoms_header + 2
        lines.insert(atoms_start - 1, "\n")

    end = atoms_start
    for i in range(atoms_start, len(lines)):
        if not is_numeric_line(lines[i]):
            end = i
            break

    atom_lines = [
        f"{a['id']} {a['mol']} {a['type']} {a['q']} "
        f"{a['x']} {a['y']} {a['z']} "
        f"{a['ix']} {a['iy']} {a['iz']}\n"
        for a in atoms
    ]

    lines[atoms_start:end] = atom_lines

    with open(outname, "w") as f:
        f.writelines(lines)

# ============================================================
# One Ov + Oi event
# ============================================================
def process_event(atoms_in, box, origin, rng):
    atoms = deepcopy_atoms(atoms_in)

    for a in atoms:
        if a["type"] == TYPE_OV_CORE_AL:
            a["type"] = TYPE_AL
        elif a["type"] == TYPE_OI:
            a["type"] = TYPE_O

    o_indices = [i for i, a in enumerate(atoms) if a["type"] == TYPE_O]
    del_idx = int(rng.choice(o_indices))
    Ov_pos = np.array([atoms[del_idx][k] for k in ("x", "y", "z")])
    atoms.pop(del_idx)

    al_indices = [i for i, a in enumerate(atoms) if a["type"] == TYPE_AL]
    al_pos = np.array([[atoms[i][k] for k in ("x", "y", "z")] for i in al_indices])
    disp = minimum_image(al_pos - Ov_pos, box)
    nearest3 = np.argsort(np.linalg.norm(disp, axis=1))[:3]
    for i in nearest3:
        atoms[al_indices[i]]["type"] = TYPE_OV_CORE_AL

    all_pos = np.array([[a["x"], a["y"], a["z"]] for a in atoms])

    while True:
        Oi_pos = origin + rng.random(3) * box
        if np.linalg.norm(minimum_image(Oi_pos - Ov_pos, box)) < MIN_DIST:
            continue
        if np.all(np.linalg.norm(minimum_image(all_pos - Oi_pos, box), axis=1) >= MIN_DIST):
            break

    new_id = max(a["id"] for a in atoms) + 1
    atoms.append({
        "id": new_id,
        "mol": 0,
        "type": TYPE_OI,
        "q": -0.945,
        "x": float(Oi_pos[0]),
        "y": float(Oi_pos[1]),
        "z": float(Oi_pos[2]),
        "ix": 0,
        "iy": 0,
        "iz": 0,
    })

    d = float(np.linalg.norm(minimum_image(Oi_pos - Ov_pos, box)))
    return atoms, Ov_pos, Oi_pos, d

# ============================================================
# Main
# ============================================================
def main():
    rng = np.random.default_rng(123)
    event_id = 0
    summary = []

    files = sorted(glob.glob(FILE_PATTERN))
    if not files:
        raise FileNotFoundError(f"No files matched pattern: {FILE_PATTERN}")

    for f in files:
        atoms0, box, origin, lines = read_lammps_data(f)

        for _ in range(N_EVENTS_PER_SAMPLE):
            event_id += 1
            atoms_mod, Ov_pos, Oi_pos, d = process_event(atoms0, box, origin, rng)

            outname = f"{OUT_DIR}/event_{event_id:03d}_OvOi.data"
            write_lammps_data(lines, atoms_mod, outname)

            summary.append({
                "event_id": event_id,
                "Ov_x": Ov_pos[0],
                "Ov_y": Ov_pos[1],
                "Ov_z": Ov_pos[2],
                "Oi_x": Oi_pos[0],
                "Oi_y": Oi_pos[1],
                "Oi_z": Oi_pos[2],
                "Ov_Oi_dist": d,
            })

    pd.DataFrame(summary).to_csv(SUMMARY_CSV, index=False)
    print(f"[OK] Generated {event_id} events")
    print(f"[OK] Summary written to {SUMMARY_CSV}")

if __name__ == "__main__":
    main()

[OK] Generated 100 events
[OK] Summary written to OvOi_event_summary.csv
